# Warehouse Grain Investigation – Duplicate Key Investigation


> **Purpose**
> This notebook documents a data quality investigation performed while completing the FlyRank ML-04 assignment.
>
> The objective was to verify that the documented warehouse grain
> `(report_date × client_hash_id × content_hash_id)`
> was consistently enforced across the warehouse.
>
> This notebook is separate from the assignment submission and serves only as documentation of the investigation.

## Objective

Validate the documented warehouse grain across the dataset and investigate any observed violations of the expected primary key uniqueness.



>  All SQL queries and outputs are preserved exactly as executed during the investigation. Markdown has been added only to document the investigation process and observations.



## Environment Setup

Initialize the DuckDB connection and configure access to the warehouse dataset.

In [3]:
import duckdb
from getpass import getpass

token = getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

Enter your Hugging Face READ token: ··········


## Expected Warehouse Grain

The documented primary key grain for the `fact_content_daily_performance` warehouse table is:

`report_date × client_hash_id × content_hash_id`

Under this contract, each row should represent a unique page-day observation for a specific client and content item. No two rows should share the same combination of `report_date`, `client_hash_id`, and `content_hash_id`.

## Baseline Verification

During feature development, the March 2026 partition (`month = '2026-03'`) was selected as the reference analysis window. Before proceeding with full warehouse checks, the primary key grain was verified within this baseline partition.

### Question
Is the primary key grain enforced without duplicate combinations in the March 2026 analysis window?

### Rationale
Verify whether `(report_date, client_hash_id, content_hash_id)` uniquely identifies rows in the March 2026 reference window.

In [4]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,cnt


### Observation
The March 2026 verification returned no duplicate keys, confirming that the primary key grain holds within the reference analysis window.

## Warehouse-wide Grain Verification

To confirm whether the primary key contract holds across the full historical dataset, baseline checks were extended to all partitions in the warehouse.

### Question
What is the total row volume across the entire warehouse dataset?

### Rationale
Establish the overall record count prior to performing group-level uniqueness checks.

In [5]:
con.sql(f"""
SELECT COUNT(*) AS rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┐
│   rows   │
│  int64   │
├──────────┤
│ 78835655 │
└──────────┘

### Observation
The query confirmed the total record volume present across all historical parquet partitions in the warehouse.

### Question
What is the date range covered by the dataset?

### Rationale
Verify the total reporting period and dataset boundaries.

In [6]:
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┐
│ start_date │  end_date  │ total_rows │
│    date    │    date    │   int64    │
├────────────┼────────────┼────────────┤
│ 2025-01-27 │ 2026-06-30 │   78835655 │
└────────────┴────────────┴────────────┘

### Observation
The dataset spans the historical date range from early 2025 through mid-2026.

### Question
Does the primary key grain hold when evaluated across the entire warehouse dataset?

### Rationale
Check for duplicate composite key combinations `(report_date, client_hash_id, content_hash_id)` across all table partitions.

In [7]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬───────┐
│ report_date │     client_hash_id      │     content_hash_id      │  cnt  │
│    date     │         varchar         │         varchar          │ int64 │
├─────────────┼─────────────────────────┼──────────────────────────┼───────┤
│ 2026-06-13  │ client_8ddc46da5414ffd8 │ content_1d06a2c99a935e49 │     2 │
│ 2026-06-16  │ client_9c26c096d6e57253 │ content_70b02f5039a882c2 │     2 │
│ 2026-06-14  │ client_1a8bf67cad4ee525 │ content_3084acbc2aca184b │     2 │
│ 2026-06-21  │ client_1a730cb2640a1abf │ content_62a6c7628cfd006f │     2 │
│ 2026-06-22  │ client_1a8bf67cad4ee525 │ content_6f83f0f309476a88 │     2 │
└─────────────┴─────────────────────────┴──────────────────────────┴───────┘

### Observation
The warehouse-wide query returned duplicate entries for the expected composite key, revealing an unexpected primary key grain violation in the warehouse dataset.

## Investigation Steps

Following the initial discovery of duplicate primary keys, a series of targeted queries was executed to isolate the root cause, determine partition boundaries, and analyze row structure.

### 1. Duplicate Key Count

#### Question
How many duplicate composite key combinations exist across the warehouse dataset?

#### Rationale
Quantify the overall scale of the grain violation.

In [8]:
con.sql(f"""
SELECT
    COUNT(*) AS duplicate_keys
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS cnt
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ duplicate_keys │
│     int64      │
├────────────────┤
│           6390 │
└────────────────┘

#### Observation
The query confirmed that duplicate key combinations exist across the warehouse dataset.

### 2. Partition & Monthly Isolation

#### Question
Are key duplicates distributed across all historical months or isolated to specific monthly partitions?

#### Rationale
Group duplicate key counts by partition month (`month`) to identify when the anomaly began.

In [9]:
con.sql(f"""
SELECT
    month,
    COUNT(*) AS duplicate_keys
FROM (
    SELECT
        month,
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS cnt
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    GROUP BY
        month,
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
)
GROUP BY month
ORDER BY month;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,duplicate_keys
0,2026-06,6390


#### Observation
Grouping by partition month isolated all duplicate key combinations exclusively to the June 2026 partition. All prior monthly partitions (January 2025 through May 2026) returned zero duplicates.

### 3. Duplication Multiplicity Analysis

#### Question
How many times is each duplicate key repeated?

#### Rationale
Determine the frequency distribution of row counts (`cnt`) per composite key to assess duplication symmetry.

In [10]:
con.sql(f"""
SELECT
    cnt,
    COUNT(*) AS num_keys
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS cnt
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
)
WHERE cnt > 1
GROUP BY cnt
ORDER BY cnt;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,cnt,num_keys
0,2,6390


#### Observation
The frequency distribution showed that every affected composite key in the June 2026 partition was duplicated exactly twice.

### 4. Sample Duplicate Record Inspection

#### Question
What do duplicate record pairs look like at an individual column value level?

#### Rationale
Inspect all columns for a specific duplicate composite key pair in June 2026 to evaluate field differences.

In [11]:
con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date = DATE '2026-06-15'
  AND client_hash_id = 'client_a2eeb8899886adde'
  AND content_hash_id = 'content_9ce5ba9ac4527940';
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-15,client_a2eeb8899886adde,content_9ce5ba9ac4527940,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-15,client_a2eeb8899886adde,content_9ce5ba9ac4527940,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


#### Rationale (Transposed View)
Transpose the duplicate record pair to allow direct side-by-side comparison of all column attributes.

In [12]:
df = con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date = DATE '2026-06-15'
  AND client_hash_id = 'client_a2eeb8899886adde'
  AND content_hash_id = 'content_9ce5ba9ac4527940';
""").df()

print(df.T)


                                                 0                         1
report_date                    2026-06-15 00:00:00       2026-06-15 00:00:00
client_hash_id             client_a2eeb8899886adde   client_a2eeb8899886adde
content_hash_id           content_9ce5ba9ac4527940  content_9ce5ba9ac4527940
client_has_gsc                                True                      True
client_has_ga4                                True                      True
gsc_data_available                           False                     False
ga4_data_available                           False                     False
gsc_impressions                                  0                         0
gsc_clicks                                       0                         0
gsc_sum_position                                 0                         0
gsc_avg_position                               NaN                       NaN
ga4_pageviews                                    0                         0

#### Observation
The sample inspection confirmed that for a given reporting date in June 2026, two separate rows exist for the exact same client and content ID.

### 5. Full-Row MD5 Duplication Verification

#### Question
Are duplicate key pairs byte-identical line duplicates across all table columns?

#### Rationale
Concatenate and hash all column values per row using `MD5` to test whether rows are completely identical across every attribute.

In [13]:
con.sql(f"""
SELECT
    md5(concat_ws('|',
        CAST(report_date AS VARCHAR),
        client_hash_id,
        content_hash_id,
        CAST(client_has_gsc AS VARCHAR),
        CAST(client_has_ga4 AS VARCHAR),
        CAST(gsc_data_available AS VARCHAR),
        CAST(ga4_data_available AS VARCHAR),
        CAST(gsc_impressions AS VARCHAR),
        CAST(gsc_clicks AS VARCHAR),
        CAST(gsc_sum_position AS VARCHAR),
        CAST(gsc_avg_position AS VARCHAR),
        CAST(ga4_pageviews AS VARCHAR),
        CAST(ga4_sessions AS VARCHAR),
        CAST(ga4_users AS VARCHAR),
        CAST(ga4_engaged_sessions AS VARCHAR),
        CAST(ga4_total_engagement_sec AS VARCHAR),
        CAST(sessions_organic AS VARCHAR),
        CAST(sessions_direct AS VARCHAR),
        CAST(sessions_referral AS VARCHAR),
        CAST(sessions_social AS VARCHAR),
        CAST(sessions_paid AS VARCHAR),
        CAST(sessions_ai AS VARCHAR),
        CAST(ai_chatgpt AS VARCHAR),
        CAST(ai_perplexity AS VARCHAR),
        CAST(ai_gemini AS VARCHAR),
        CAST(ai_copilot AS VARCHAR),
        CAST(ai_claude AS VARCHAR),
        CAST(ai_meta AS VARCHAR),
        CAST(ai_other AS VARCHAR),
        CAST(scroll_events AS VARCHAR),
        month
    )) AS row_hash,
    COUNT(*) AS cnt
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
GROUP BY row_hash
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 10;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_hash,cnt
0,e8d2f943175ff3b5edab11dd0c355f42,2
1,08be6d54009a336f6ce1330d7e753b2a,2
2,35d808aadc28c0c14807a8a08247ea03,2
3,1787c947966538cab6f00700332c8726,2
4,8cec2371c697a5806db8b4987c8c59f3,2
5,643f204146ada11419ad7faebef9078c,2
6,6560d692be698cfbd15c3bbf3a0eb6d6,2
7,8afffe82d3898601857db93ed82d6f69,2
8,92655661e44f62a2ce87a719882e6e4e,2
9,df435505df219e3c3e81251f982d1631,2


#### Observation
The MD5 hash verification across all dataset columns returned duplicated row hashes with a count of 2, confirming that the duplicate key records in the June 2026 partition are byte-identical full-row duplicates across all columns.

## Findings

The key empirical findings from the investigation are summarized below:

- **Baseline Grain Enforcement:** The documented primary key grain `(report_date × client_hash_id × content_hash_id)` is enforced without duplicates across all historical partitions from January 2025 through May 2026.
- **Partition Isolation:** Primary key duplication occurs exclusively within the June 2026 partition (`month = '2026-06'`).
- **Uniform Multiplicity:** All affected primary key combinations in the June 2026 partition are duplicated with a uniform multiplicity of exactly two.
- **Byte-Identical Line Duplicates:** MD5 hashing across all table columns confirmed that duplicate rows are completely identical across all attributes and metrics.

## Conclusion

1. The investigation isolated the observed duplicate-key combinations strictly to the June 2026 partition.
2. The assignment model development safely proceeded using the recommended March 2026 analysis window, which was empirically verified to be duplicate-free.
3. The observation was reported to the FlyRank team for clarification of the data discrepancy.